# Jour 2 — Analyse de devis avec pandas

*Formation Python pour l'analyse de données — sujet*

On analyse des **devis** : un client, un produit, une date d'édition, un montant, et une réponse
(**accepté** ou **refusé**). Les données sont **générées au hasard** par la première cellule de code :
rien à télécharger, et tout le monde a exactement les mêmes chiffres (graine fixe).

Équivalences SAS utiles : `PROC MEANS` → `groupby().agg()` / `describe()`,
`PROC FREQ` → `value_counts()` / `pd.crosstab()`, `PROC TABULATE` → `pivot_table()`,
`PROC SGPLOT VBOX` → `boxplot()`.

## 0. Générer les données

Exécutez la cellule suivante sans la modifier.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(2026)          # graine fixe : mêmes données pour tout le monde
n = 6_000

produits = ["AUTO", "HABITATION", "SANTE", "PREVOYANCE"]
produit = rng.choice(produits, n, p=[0.40, 0.30, 0.20, 0.10])

# montant annuel du devis : ordre de grandeur différent selon le produit
moyenne = pd.Series(produit).map({"AUTO": 750, "HABITATION": 420, "SANTE": 1300, "PREVOYANCE": 600})
montant = np.round(rng.gamma(4, moyenne / 4), 2)

# dates d'édition sur 22 mois, plus de devis en septembre-octobre (rentrée)
jours = pd.date_range("2025-01-01", "2026-10-31", freq="D")
poids = np.where(jours.month.isin([9, 10]), 1.8, 1.0)
date_edition = rng.choice(jours, n, p=poids / poids.sum())

# probabilité d'acceptation : plus faible quand le devis est cher pour son produit
p_accepte = 0.75 - 0.35 * (montant / moyenne.to_numpy() - 1).clip(-1, 1.5)
statut = np.where(rng.random(n) < p_accepte.clip(0.05, 0.95), "ACCEPTE", "REFUSE")

devis = pd.DataFrame({
    "num_client": "CL" + pd.Series(rng.integers(1, 2_500, n)).astype(str).str.zfill(5),
    "produit": produit,
    "date_edition": date_edition,
    "montant": montant,
    "statut": statut,
})

# quelques anomalies de saisie, comme dans la vraie vie
devis.loc[rng.choice(n, 6, replace=False), "montant"] *= -1          # montants négatifs
devis.loc[rng.choice(n, 25, replace=False), "statut"] = "Refusé "     # saisie manuelle
devis = pd.concat([devis, devis.sample(15, random_state=1)])          # doublons
devis = devis.sort_values("date_edition").reset_index(drop=True)

devis.head()

,num_client,produit,date_edition,montant,statut
0,CL01607,HABITATION,2025-01-01,570.76,ACCEPTE
1,CL02014,SANTE,2025-01-01,1857.78,REFUSE
2,CL02128,AUTO,2025-01-01,156.48,ACCEPTE
3,CL00307,SANTE,2025-01-01,724.73,ACCEPTE
4,CL01877,SANTE,2025-01-01,1535.64,REFUSE


## 1. Premier coup d'œil  ★☆☆

Combien de lignes et de colonnes ? Quels types de colonnes (`dtypes`) ?
Affichez les statistiques de `montant` (`describe`) et les valeurs distinctes de `produit` et `statut`
(`value_counts`). Repérez-vous quelque chose de bizarre ?

In [ ]:
# À vous

## 2. Nettoyer  ★☆☆

1. Supprimez les doublons exacts (`drop_duplicates`) : combien de lignes en moins ?
2. Harmonisez `statut` pour n'avoir que `ACCEPTE` et `REFUSE`
   (indice : `.str.strip()`, `.str.upper()`, puis `.replace({"REFUSÉ": "REFUSE"})`).
3. Écartez les montants négatifs (gardez-les de côté dans `anomalies` pour les signaler).
4. Passez `produit` et `statut` en type `category`.

In [ ]:
# À vous

## 3. Tableau par produit (PROC MEANS)  ★☆☆

Par produit : nombre de devis, montant moyen, médian, minimum et maximum
(`groupby(...).agg(...)` avec des noms de colonnes lisibles), trié par montant moyen décroissant.

In [ ]:
# À vous

## 4. Taux d'acceptation (PROC FREQ)  ★☆☆

1. Taux d'acceptation global, en % (indice : `(devis["statut"] == "ACCEPTE").mean()`).
2. Tableau croisé produit × statut, en effectifs puis en % par ligne (`pd.crosstab`, `normalize="index"`).
3. Quel produit a le meilleur taux d'acceptation ?

In [ ]:
# À vous

## 5. Comparer avec des boxplots  ★☆☆

1. Boxplot du montant par produit (`devis.boxplot(column=..., by=...)`).
2. Avec seaborn : boxplot du montant par produit, coloré par statut (`hue=`).
3. Que remarquez-vous sur les montants des devis refusés ?

In [ ]:
# À vous

## 6. Le prix fait-il refuser ?  ★★☆

Découpez le montant en 4 tranches de même effectif **à l'intérieur de chaque produit**
(`groupby("produit")["montant"].transform(lambda s: pd.qcut(s, 4, labels=["Q1", "Q2", "Q3", "Q4"]))`),
puis calculez le taux d'acceptation par produit et par tranche (`pivot_table`).
Le taux baisse-t-il quand le montant monte ?

In [ ]:
# À vous

## 7. Évolution dans le temps  ★★☆

Nombre de devis et taux d'acceptation **par mois** (`resample("MS", on="date_edition")` ou
`groupby(devis["date_edition"].dt.to_period("M"))`). Tracez les deux courbes.
Voyez-vous un effet saisonnier ?

In [ ]:
# À vous

## 8. Les clients  ★★☆

1. Combien de clients différents (`nunique`) ? Combien de devis par client en moyenne ?
2. Les 10 clients qui ont le plus de devis **acceptés**, avec le montant total accepté.
3. Combien de clients n'ont **jamais** accepté un devis ?

In [ ]:
# À vous

## 9. Synthèse à exporter  ★★★

Construisez un tableau par produit avec : nombre de devis, taux d'acceptation (%),
montant moyen des devis acceptés, montant moyen des devis refusés, chiffre d'affaires accepté (somme).
Exportez-le en Excel (`to_excel("synthese_devis.xlsx")`) et en CSV au format français
(`sep=";"`, `decimal=","`, `encoding="utf-8-sig"`).

In [7]:
# À vous
moyenne = pd.Series(produit).map({"AUTO": 750, "HABITATION": 420, "SANTE": 1300, "PREVOYANCE": 600})
moyenne

0       750
1       420
2       420
3       750
4       750
       ... 
5995    420
5996    750
5997    750
5998    750
5999    600
Length: 6000, dtype: int64

In [8]:
produit

array(['AUTO', 'HABITATION', 'HABITATION', ..., 'AUTO', 'AUTO',
       'PREVOYANCE'], shape=(6000,), dtype='<U10')